In [ ]:
%%writefile core_engine.py
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
class AdvancedMatcher:
    def __init__(self, items, mode="count"):
        if not items or any(not item["content"].strip() for item in items):
            raise ValueError("Items cannot be empty or have blank content")
        self.items = items
        self.mode = mode
        corpus = [item["content"] for item in items]
        if mode == "count":
            self.vectorizer = CountVectorizer()
            self.matrix = self.vectorizer.fit_transform(corpus)
        elif mode == "neural":
            from sentence_transformers import SentenceTransformer
            self.encoder = SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.validate_length(corpus)
            self.matrix = self.encoder.encode(corpus, normalize_embeddings=True)
        else:
            raise ValueError("Unsupported mode specified")
    def validate_length(self, texts):
        for txt in texts:
            tokens = self.encoder.tokenizer(txt, truncation=False)["input_ids"]
            if len(tokens) > self.encoder.max_seq_length:
                raise ValueError("Text exceeds maximum sequence length")
    def query_match(self, prompt, top_n=3, filters=None):
        if not prompt.strip() or top_n <= 0:
            return []
        if self.mode == "count":
            vec = self.vectorizer.transform([prompt])
            if vec.nnz == 0:
                return []
            ratings = (vec @ self.matrix.T).toarray()[0]
        else:
            self.validate_length([prompt])
            vec = self.encoder.encode([prompt], normalize_embeddings=True)[0]
            ratings = self.matrix @ vec
        valid_indices = []
        for idx, item in enumerate(self.items):
            if all(item.get(k) == v for k, v in (filters or {}).items()):
                valid_indices.append(idx)
        ordered = sorted(valid_indices, key=lambda idx: (-float(ratings[idx]), idx))
        return [(self.items[idx], float(ratings[idx])) for idx in ordered[:top_n]]
def render_matches(matches):
    if not matches:
        print("No matching entries found.")
        return
    for record, score in matches:
        print(record["code"], f"relevance={score:.3f}")
        print(record["content"])

In [ ]:
%%writefile support_assistant.py

from core_engine import AdvancedMatcher

KNOWLEDGE_BASE = [
    {
        "code": "kb:refund_delay",
        "prompt": "Online order cancel karne par refund kitne din mein milta hai?",
        "reply": "Sir, item warehouse par receive hone ke baad 3 se 5 working days mein amount refund ho jati hai.",
        "origin": "finance_policy.txt"
    },
    {
        "code": "kb:delivery_time",
        "prompt": "Order place karne ke baad delivery mein kitna time lagta hai?",
        "reply": "Standard shipping 2 se 4 business days leti hai, jabke express delivery 24 hours mein hoti hai.",
        "origin": "shipping_policy.txt"
    },
    {
        "code": "kb:warranty_claim",
        "prompt": "Electronics product kharab hone par warranty claim kaise karein?",
        "reply": "Aap official warranty card aur purchase invoice ke sath nearest service center visit kar sakte hain.",
        "origin": "warranty_policy.txt"
    }
]
def create_assistant(mode="count"):
    records = []
    for entry in KNOWLEDGE_BASE:
        rec = entry.copy()
        rec["content"] = entry["prompt"] + " " + entry["reply"]
        records.append(rec)
    return AdvancedMatcher(records, mode)
def get_response(bot, user_query):
    if not user_query.strip():
        name_msg = "Kindly type a valid inquiry."
        return name_msg
    for entry in KNOWLEDGE_BASE:
        if user_query.strip().lower() == entry["prompt"].lower():
            return entry["reply"] + " [Source: " + entry["origin"] + "]"
    hits = bot.query_match(user_query)
    if len(hits) == 0:
        return "Is query ka koi mutabiq jawab knowledge base mein mojood nahi hai."
    output_lines = ["Related inquiries found:"]
    for item, sc in hits:
        output_lines.append(item["prompt"] + f" (Score: {sc:.3f})")
    return "\n".join(output_lines)
if __name__ == "__main__":
    import argparse
    ap = argparse.ArgumentParser()
    ap.add_argument("--mode", choices=["count", "neural"], default="count")
    parsed_args = ap.parse_args()
    assistant_bot = create_assistant(parsed_args.mode)
    print("E-Commerce Support Assistant Active")
    print("Type 'quit' to exit session.")
    while True:
        try:
            user_input = input("Client: ")
        except (EOFError, KeyboardInterrupt):
            break
        if user_input.strip().lower() == "quit":
            break
        response_text = get_response(assistant_bot, user_input)
        print("Agent:", response_text)

In [ ]:
!ls

In [ ]:
%%writefile segment_builder.py
import re
def extract_spans(txt):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", txt)]
def token_windows(txt, chunk_size=35, step_overlap=8, base_shift=0):
    if chunk_size <= 0 or step_overlap < 0 or step_overlap >= chunk_size:
        raise ValueError("Bad parameters for token windows")
    spans = extract_spans(txt)
    segments = []
    for index in range(0, len(spans), chunk_size - step_overlap):
        end_idx = min(index + chunk_size, len(spans))
        segments.append((base_shift + spans[index][0], base_shift + spans[end_idx - 1][1]))
        if end_idx == len(spans):
            break
    return segments
def sentence_segments(txt, chunk_size=35):
    if chunk_size <= 0:
        raise ValueError("Size must be positive integer")
    breakpoints = [0] + [m.end() for m in re.finditer(r"[.!?]\s+", txt)] + [len(txt)]
    segments, current_pending, word_count = [], None, 0
    for start_p, end_p in zip(breakpoints, breakpoints[1:]):
        w_len = len(txt[start_p:end_p].split())
        if not w_len:
            continue
        if w_len > chunk_size:
            if current_pending:
                segments.append(current_pending)
            segments.extend(token_windows(txt[start_p:end_p], chunk_size, 0, start_p))
            current_pending, word_count = None, 0
        elif current_pending and word_count + w_len <= chunk_size:
            current_pending, word_count = (current_pending[0], end_p), word_count + w_len
        else:
            if current_pending:
                segments.append(current_pending)
            current_pending, word_count = (start_p, end_p), w_len
    if current_pending:
        segments.append(current_pending)
    return segments
def build_all_segments(library, strat_mode):
    collected = []
    for doc_key, content_str in library.items():
        matched_spans = sentence_segments(content_str) if strat_mode == "sentence" else token_windows(content_str, 35, int(strat_mode))
        for idx, (st, en) in enumerate(matched_spans):
            collected.append({
                "code": f"{doc_key}:{idx}", "document": doc_key,
                "start": st, "end": en, "content": content_str[st:en]
            })
    return collected
if __name__ == "__main__":
    assert token_windows("") == []
    assert token_windows("sample test") == [(0, 11)]
    try:
        token_windows("test check", 2, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Overlap validation failed")


In [ ]:
!python segment_builder.py

In [ ]:
%%writefile sample_test_runner.py
from segment_builder import token_windows, sentence_segments
sample_body = ("Warranty claims require a valid invoice. Service is denied for physical damage "
               "or unauthorized modifications. Contact technical support team.")
test_cases = [
    ("token windows", token_windows(sample_body, 10, 3)),
    ("sentence segments", sentence_segments(sample_body, 10))
]
for label, spans in test_cases:
    print(label)
    for s_idx, e_idx in spans:
        print(s_idx, e_idx, repr(sample_body[s_idx:e_idx]))
        assert 0 <= s_idx < e_idx <= len(sample_body)

In [ ]:
!python sample_test_runner.py

In [ ]:
%%writefile evaluation_suite.py
from segment_builder import build_all_segments, token_windows, sentence_segments

CORPORA = {
    "shipping": (
        "Shipping Guidelines. All orders are processed within twenty-four hours "
        "of payment confirmation. Standard courier delivery takes three to five "
        "working days across major cities. Remote locations may require up to seven "
        "business days. Customers receive an automated tracking link via email and SMS "
        "once the package leaves the warehouse. If a shipment is delayed due to weather "
        "or political strikes, the logistics department will notify clients promptly. "
        "Express delivery options are available at checkout for an additional fee. "
        "Signature is mandatory upon receiving high-value electronics items. "
        "Clients must inspect packages for external damage before signing the delivery slip. "
        "Any discrepancies or missing items should be reported within forty-eight hours."),
    "security": (
        "Account Security Protocol. Every registered user must maintain a secure "
        "password consisting of letters, numbers, and symbols. Two-factor authentication "
        "is highly recommended for all financial transactions. If you suspect unauthorized "
        "activity, immediately freeze your account using the mobile app security settings. "
        "Customer service agents will never ask for your account password or PIN code. "
        "Password resets can be requested via the recovery portal linked on the login screen. "
        "Accounts failing five consecutive login attempts are temporarily locked for thirty minutes. "
        "Users must update their login credentials every ninety days to comply with security standards."),
    "returns": (
        "Return and Refund Policy. Items purchased can be returned within fourteen days "
        "of delivery if they remain unused and in original packaging. Clearance sale items "
        "and customized products are strictly non-returnable. To initiate a return, log into "
        "your customer profile and generate a return merchandise authorization label. The "
        "courier partner will pick up the parcel from your doorstep free of charge. Once "
        "the warehouse team inspects the returned goods, refunds are credited back to the "
        "original payment source within five working days."),
}
TARGET_PAIRS = {
    "shipping": ("Standard courier delivery takes three to five working days across major cities.",
                 "Remote locations may require up to seven business days."),
    "security": ("Accounts failing five consecutive login attempts are temporarily locked for thirty minutes.",
                 "Users must update their login credentials every ninety days"),
    "returns": ("Items purchased can be returned within fourteen days of delivery",
                "Clearance sale items and customized products are strictly non-returnable."),
}
if __name__ == "__main__":
    orig_counts = {k: len(v.split()) for k, v in CORPORA.items()}
    for k, v in CORPORA.items():
        print(k, "words:", orig_counts[k])
    total_words = sum(orig_counts.values())
    print()
    for mode_key, mode_name in [("0", "window 35, overlap 0"),
                                ("8", "window 35, overlap 8"),
                                ("sentence", "sentence-aware 35")]:
        segments = build_all_segments(CORPORA, mode_key)
        segment_sizes = [len(c["content"].split()) for c in segments]
        print(f"{mode_name}: total segments={len(segments)}, avg words={sum(segment_sizes)/len(segment_sizes):.1f}")

In [ ]:
!python evaluation_suite.py

In [ ]:
%%writefile embedding_eval.py
import numpy as np
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
snippets = [
    "Reset your account password through the secure recovery link.",
    "Track your active order status via the customer dashboard.",
    "Check printer cartridges when white streaks appear on paper."
]
doc_mat = embed_model.encode(snippets, normalize_embeddings=True)
query_mat = embed_model.encode(["How do I reset my credentials?"], normalize_embeddings=True)
sim_scores = (query_mat @ doc_mat.T)[0]
print("Embedding Matrix Shape:", doc_mat.shape)
for index in np.argsort(-sim_scores):
    print(index, round(float(sim_scores[index]), 3), snippets[index])
assert np.allclose(np.linalg.norm(doc_mat, axis=1), 1, atol=1e-5)

In [ ]:
!python embedding_eval.py

In [ ]:
%%writefile suggestion_test.py
from support_assistant import create_assistant
bot_engine = create_assistant("neural")
search_term = "shipping time"
print("Top suggested prompts:")
for record, sc in bot_engine.query_match(search_term, top_n=3):
    print(record["prompt"], f"({sc:.3f})")

In [ ]:
!python suggestion_test.py

In [ ]:
%%writefile semantic_validation.py
from segment_builder import build_all_segments
from evaluation_suite import CORPORA
from core_engine import AdvancedMatcher
chunk_records = build_all_segments(CORPORA, "sentence")
test_queries = [
    "How can I track my package delivery?",
    "What should I do if my account gets locked?",
    "How many days do I have to return an item?"
]
for backend_type in ["count", "neural"]:
    engine_obj = AdvancedMatcher(chunk_records, backend_type)
    print("="*60)
    print("Backend Engine:", backend_type)
    for q_text in test_queries:
        print("\nQuery:", q_text)
        hits = engine_obj.query_match(q_text, top_n=2)
        if not hits:
            print("No matches found.")
        for res_item, sc_val in hits:
            print(f"{res_item['code']} score={sc_val:.3f} bounds {res_item['start']}-{res_item['end']}")
            print(res_item['content'])

In [ ]:
!python semantic_validation.py